# Word2Vec CBOW Example

This notebook demonstrates Continuous Bag of Words (CBOW) using Gensim's `Word2Vec`.
- CBOW predicts the current word from the surrounding context words.
- The notebook also shows how to load Google's pre-trained Word2Vec vectors with `gensim.downloader`.

In [1]:
from gensim.models import Word2Vec, KeyedVectors
import gensim.downloader as api

## Prepare the training corpus

Use a small tokenized corpus so the training behavior is easy to inspect.

In [2]:
sentences = [
    ["people", "watch", "movie", "again"],
    ["people", "watch", "cricket", "again"],
    ["people", "like", "movie", "a", "lot"],
    ["people", "like", "cricket", "a", "lot"],
    ["movie", "fans", "like", "great", "stories"],
    ["cricket", "fans", "like", "great", "matches"]
]

sentences

[['people', 'watch', 'movie', 'again'],
 ['people', 'watch', 'cricket', 'again'],
 ['people', 'like', 'movie', 'a', 'lot'],
 ['people', 'like', 'cricket', 'a', 'lot'],
 ['movie', 'fans', 'like', 'great', 'stories'],
 ['cricket', 'fans', 'like', 'great', 'matches']]

## Train a CBOW model (CUSTOM MODEL)

In Gensim, `sg=0` means CBOW. Here we **create our own embeddings** by training on our corpus.

In [3]:
# CUSTOM MODEL: Train our own embeddings on the small corpus
cbow_model = Word2Vec(
    sentences=sentences,
    vector_size=50,
    window=2,
    min_count=1,
    workers=1,
    sg=0,
    epochs=200
)

In [4]:
print("Vocabulary:", list(cbow_model.wv.index_to_key))

Vocabulary: ['like', 'people', 'cricket', 'movie', 'great', 'fans', 'lot', 'a', 'again', 'watch', 'matches', 'stories']


## Inspect word vectors

Each word gets a dense embedding instead of a sparse count-based vector.

In [5]:
print("Vector for 'movie':\n", cbow_model.wv["movie"])

Vector for 'movie':
 [ 0.01524381 -0.01899547 -0.00038939  0.00730283 -0.00235973  0.01627035
  0.01881397  0.01370161 -0.00240512  0.0149936  -0.01663038  0.00601211
 -0.00910819 -0.01019033  0.00725183  0.01099635  0.01655377 -0.01064755
  0.01362958  0.012711   -0.00697515 -0.01734019  0.01207103  0.01311621
 -0.00110306 -0.01329939 -0.01406984 -0.00411102  0.00959638 -0.00731567
 -0.01863256  0.0068619   0.01008107 -0.01391526  0.00217788 -0.00452225
  0.00090457 -0.01997502  0.00575588 -0.00938816  0.00204537 -0.00283748
  0.00375639 -0.01623601 -0.00449813  0.00602783  0.01081002 -0.00483281
 -0.01873202  0.00919105]


In [6]:
print("Words similar to 'movie':")
print(cbow_model.wv.most_similar("movie", topn=5))

Words similar to 'movie':
[('a', 0.057210735976696014), ('like', 0.01894540898501873), ('again', -0.005220409017056227), ('matches', -0.016965214163064957), ('lot', -0.02945856936275959)]


## Load Google's pre-trained Word2Vec model (PRE-TRAINED MODEL)

This downloads the `word2vec-google-news-300` vectors the first time you run it. Here we **use embeddings already trained** by Google on billions of words.
It is a large download, so keep this cell optional if bandwidth or disk space is limited.

In [7]:
# PRE-TRAINED MODEL: Download embeddings trained by Google on billions of words
google_news_vectors = api.load("word2vec-google-news-300")
type(google_news_vectors)

[=========================-------------------------] 51.0% 848.5/1662.8MB downloaded

[==================================================] 100.0% 1662.8/1662.8MB downloaded


gensim.models.keyedvectors.KeyedVectors

In [8]:
# Using PRE-TRAINED MODEL: Compare with our custom model trained above
print("Vector size from Google News model:", google_news_vectors.vector_size)
print(google_news_vectors.most_similar("movie", topn=5))

Vector size from Google News model: 300
[('film', 0.8676770329475403), ('movies', 0.8013108372688293), ('films', 0.7363011837005615), ('moive', 0.6830360889434814), ('Movie', 0.6693680286407471)]


In [11]:
print(google_news_vectors.similarity("man","woman"))

0.76640123


## Summary

- CBOW predicts a target word from nearby context words.
- Word2Vec creates dense semantic vectors instead of sparse count matrices.
- Google's pre-trained vectors are useful when you want stronger general-purpose embeddings without training from scratch.

## Average Word2Vec — Representing a Sentence as One Vector

**What's happening here?**
A single word has its own vector (e.g., "movie" → 300 numbers). But what about a full sentence?

**Average Word2Vec** takes all the word vectors in a sentence and **averages them** into one single vector — this lets you represent an entire sentence as a fixed-size vector for ML tasks.

**Approach:**
1. Get the vector for each word in the sentence
2. Stack them together
3. Take the mean (average) across all words

> We'll use our **custom CBOW model** for this example.

In [12]:
import numpy as np

def average_word2vec(sentence, model):
    """
    Takes a list of words (sentence) and returns the average of their word vectors.
    Skips words not in the model's vocabulary.
    """
    vectors = []
    for word in sentence:
        if word in model.wv:          # only include words the model knows
            vectors.append(model.wv[word])
    
    if not vectors:
        return None                   # return None if no words were found
    
    return np.mean(vectors, axis=0)   # average all word vectors

# Test sentence
sentence = ["people", "like", "movie", "a", "lot"]
avg_vector = average_word2vec(sentence, cbow_model)

print("Sentence         :", sentence)
print("Average vector shape:", avg_vector.shape)   # (50,) — same as vector_size
print("Average vector   :\n", avg_vector)

Sentence         : ['people', 'like', 'movie', 'a', 'lot']
Average vector shape: (50,)
Average vector   :
 [-2.27131858e-03  1.86371931e-03 -3.60968569e-03  9.34314914e-04
 -1.85961404e-03 -3.41622159e-03  1.06645385e-02  1.06846113e-02
  6.75406336e-05  1.44959238e-04  3.23154707e-03  4.34622727e-03
 -4.47051047e-04 -5.51446201e-03  4.09653457e-03 -6.11179858e-04
  8.10332783e-03  2.23167147e-03 -6.82006311e-03 -1.88854779e-03
  7.02023413e-03 -4.79756098e-04  8.80285166e-03  7.32939970e-03
 -4.94749867e-04 -4.60354518e-03  1.47083396e-04  6.22576429e-03
 -4.12697252e-03 -5.52362017e-03 -8.89011938e-03  2.46225903e-03
  7.12997979e-03 -3.53715825e-03 -1.96007290e-03  1.17568416e-03
  1.38460351e-02 -8.73602740e-03 -4.52673435e-03 -9.29799280e-04
 -4.48196055e-03  3.39832972e-03  3.66155611e-04 -7.45351519e-03
  1.05295088e-02  4.51065181e-03 -8.14897357e-04 -3.91858676e-03
 -3.45494272e-03  4.98240348e-03]


### Compare Average Vectors of Two Sentences

Now let's compare two different sentences. Sentences with similar meaning should have **closer average vectors**.

In [13]:
from numpy.linalg import norm

sentence1 = ["people", "like", "movie", "a", "lot"]
sentence2 = ["people", "like", "cricket", "a", "lot"]
sentence3 = ["movie", "fans", "like", "great", "stories"]

vec1 = average_word2vec(sentence1, cbow_model)
vec2 = average_word2vec(sentence2, cbow_model)
vec3 = average_word2vec(sentence3, cbow_model)

# Cosine similarity: 1.0 = identical, 0.0 = completely different
def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (norm(v1) * norm(v2))

print("Similarity (sentence1 vs sentence2):", round(cosine_similarity(vec1, vec2), 4))
# sentence1 and sentence2 share most words → should be HIGH similarity

print("Similarity (sentence1 vs sentence3):", round(cosine_similarity(vec1, vec3), 4))
# sentence1 and sentence3 share fewer words → should be LOWER similarity

Similarity (sentence1 vs sentence2): 0.8015
Similarity (sentence1 vs sentence3): 0.4967
